### ??? ??

### ???

In [ ]:
import FinanceDataReader as fdr
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

# CSV ???? ?? ?? ?? ????
kosdak = pd.read_csv('KOSDAQ_result.csv')
# stock_codes = kosdak['Code'].astype(str).str.zfill(6).tolist()  # ?? ?? ?? ?? (0?? ???)

# ?? ?? ?? ??
stock_codes = kosdak['Code'].astype(str).str.zfill(6).tolist()

# ?? ?? ?? ?? ??
print("Stock Codes:")
print(stock_codes)

# ? ??? ??? ??? ????
models = {}

for code in stock_codes:
    try:
        df = fdr.DataReader(code).dropna()  # ??? ???? ? ??? ??
        if not df.empty:
            # ?? ??? ??
            # df['Change'] = df['Close'].pct_change()
            df = df.dropna()
            # EMA ?? ? MACD, Signal Line ??
            df['ema_10'] = df['Close'].ewm(span=10, adjust=False).mean()
            df['ema_12'] = df['Close'].ewm(span=12, adjust=False).mean()
            df['ema_26'] = df['Close'].ewm(span=26, adjust=False).mean()
            df['ema_30'] = df['Close'].ewm(span=30, adjust=False).mean()
            df['MACD'] = df['ema_12'] - df['ema_26']
            df['Signal Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
            # Signals ??
            signals = 2  # ? ?? ??? ??? ?? ???? ?? ??? 2? ??
#             for i in range(1, len(df)):
#                 if ((df['MACD'].iloc[i-1] < df['Signal Line'].iloc[i-1]) and (df['MACD'].iloc[i] > df['Signal Line'].iloc[i])):# or \
#                #    ((df['ema_10'].iloc[i-1] < df['ema_30'].iloc[i-1]) and (df['ema_10'].iloc[i] > df['ema_30'].iloc[i])):
#                     signals.append(1)  # ?? ??
#                 elif ((df['MACD'].iloc[i-1] > df['Signal Line'].iloc[i-1]) and (df['MACD'].iloc[i] < df['Signal Line'].iloc[i])):# or \
#               #       ((df['ema_10'].iloc[i-1] > df['ema_30'].iloc[i-1]) and (df['ema_10'].iloc[i] < df['ema_30'].iloc[i])):
#                     signals.append(0)  # ?? ??
#                 else:
#                     signals.append(2)  # ?? ??
            df['Signal'] = signals
            # Position ? ???
            df['Position'] = 0
            for i in range(0, len(df), 100):
                avg_volume = df.iloc[i:i+100]['Volume'].mean()
                df.iloc[i:i+100, df.columns.get_loc('Position')] = np.where(df.iloc[i:i+100]['Volume'] > avg_volume, 1, 0)
                
            # ??/?? ?? ????
            df['Signal'] = np.where((df['Position'] == 1) & (df['Change'] < 0) & (df['Signal'] == 2), 1, df['Signal'])  # ??
            df['Signal'] = np.where((df['Position'] == 1) & (df['Change'] > 0) & (df['Signal'] == 2), 0, df['Signal'])  # ??
            
            # ??? ?? ??
            window = 5  # ???? ??
            df['SMA'] = df['Close'].rolling(window=window).mean()  # ?? ????
            df['STD'] = df['Close'].rolling(window=window).std()   # ????
            df['Upper Band'] = df['SMA'] + (df['STD'] * 2)       # ?? ??
            df['Lower Band'] = df['SMA'] - (df['STD'] * 2)      # ?? ??
            
            # ??? ??
            df = df.dropna()
            # Position2 ? Position3 ? ???
            df[['Position2', 'Position3']] = 0
            # ??? ? ??? ??
            df['Position2'] = np.where(df['Close'] >= df['Upper Band'], 1, df['Position2'])  # ??? ??
            df['Position3'] = np.where(df['Close'] <= df['Lower Band'], 1, df['Position3'])  # ??? ??
            # ??/?? ?? ????
            df['Signal'] = np.where((df['Position2'] == 1) & (df['Signal'] == 2), 0, df['Signal'])  # ?? ??
            df['Signal'] = np.where((df['Position3'] == 1) & (df['Signal'] == 2), 1, df['Signal'])  # ?? ??

            
            # ???? ? ??
            columns_to_remove = ['ema_10', 'ema_12', 'ema_26', 'ema_30', 'MACD', 'Signal Line', 'Position', 'SMA', 'STD', 'Upper Band', 'Lower Band', 'Position2', 'Position3']
            df = df.drop(columns=columns_to_remove)

            df.drop(columns=['Change'],inplace=True) # Recall? ?? ?? ???? ? 
            
            # ?? NaN ? ??
            df = df.dropna()
            X = df.iloc[:,:-1]
            y = df['Signal']
            df['Signal'] = df['Signal'].shift(-1)
            df = df.dropna(subset=['Signal'])
            
             df.drop(['2024-08-22'],inplace=True)

            
            # ??? ?? (?? ???? ??? ???)
            X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.3, shuffle=True, random_state=42, stratify=y)
            # XGBoost ?? ?? ?? ? ??
            model = xgb.XGBClassifier(
                objective='multi:softmax',
                eval_metric='mlogloss',
                learning_rate=0.1,
                n_estimators=100,
                max_depth=5,
                subsample=0.8,
                colsample_bytree=0.8,
                num_class=3,  # ??? ? (??, ??, ?? ??)
                random_state=42
            )
            model.fit(X_train, Y_train)
            # ??? ???? ?? ?? ??
            Y_pred = model.predict(X_test)
            # ?? ??
            accuracy = accuracy_score(Y_test, Y_pred)
            conf_matrix = confusion_matrix(Y_test, Y_pred)
            class_report = classification_report(Y_test, Y_pred)
            # ?? ??
            model_filename = f'model_{code}.joblib'
            joblib.dump(model, model_filename)
            models[code] = model_filename
            # ?? ??
            print(f"Model for code {code} saved as {model_filename}")
            print(f"Accuracy for code {code}: {accuracy}")
            print(f"Confusion Matrix for code {code}:\n{conf_matrix}")
            print(f"Classification Report for code {code}:\n{class_report}")
    except Exception as e:
        print(f"Error processing code {code}: {e}")


### ??? ?? ??

In [ ]:
import joblib
import numpy as np
import pandas as pd
import FinanceDataReader as fdr
from collections import defaultdict

def load_model_and_predict(company_code, recent_data):
    # ?? ??? ?? ?? ???
    model_filename = f'model_{company_code}.joblib'
    
    try:
        # ?? ????
        model = joblib.load(model_filename)
        
        # ?? ??
        predicted_signal = model.predict(recent_data)
        
        # ?? ?? ??
        signal_map = {0: 'Sell', 1: 'Buy', 2: 'No Signal'}
        return predicted_signal[0], signal_map[predicted_signal[0]]
    
    except FileNotFoundError:
        return None, f"Model file for code {company_code} not found."
    except Exception as e:
        return None, f"Error during prediction: {e}"

def display_stock_codes(codes):
    print("Available Stock Codes:")
    for i, code in enumerate(codes):
        if i > 0 and i % 10 == 0:
            print()  # ???
        print(code, end=' ')
    print()  # ??? ???

def main():
    # CSV ???? ?? ?? ?? ????
    kosdak = pd.read_csv('KOSDAQ_result.csv')
    
    # ?? ?? ?? ??
    stock_codes = kosdak['Code'].astype(str).str.zfill(6).tolist()
    
    # ??? ?? ?? ?? ??
    display_stock_codes(stock_codes)
    
    # ? ???? ??? ??? ????
    signal_counts = defaultdict(int)
    
    # ?? ?? ??? ?? ?? ??
    for company_code in stock_codes:
        # ?? ?? ??? ??? ????
        df = fdr.DataReader(company_code).dropna()
        recent_data = df.iloc[-1][['Open', 'High', 'Low', 'Close', 'Volume']].values.reshape(1, -1) # , 'Change'
        
        # ?? ??
        signal, result = load_model_and_predict(company_code, recent_data)
        
        if signal is not None:
            signal_counts[signal] += 1
            print(f'For company code {company_code}, the predicted signal for the next day is: {result}')
        else:
            print(f'Error for company code {company_code}: {result}')
    
    # ? ???? ?? ??
    print("\nSignal Counts Summary:")
    print(f"Sell (0): {signal_counts[0]}")
    print(f"Buy (1): {signal_counts[1]}")
    print(f"No Signal (2): {signal_counts[2]}")

if __name__ == "__main__":
    main()
